# Qwen3-8B — B 실험: 최대5문장 발췌

**A100에서 새 R2 930응답만 생성합니다.** 기존 최초 답변과 같은 문항의 R1 S0/S1 결과를 재사용합니다.
R1=정답 선택지만 제시, R2=동일 주장+공개 풀이의 짧은 연속 발췌. S0=방어 없음, S1=기존 전체 방어입니다.

**이번 max5 노트북과 `sycophancy_qwen_8b_evidenceB_max5_inputs.zip`을 사용하세요.**
A100 → 1번 설치 → 세션 다시 시작 → 2~8번. 설정 변경은 필요 없습니다.

전체155문항 중 **초기 오답77문항**으로 수정률, **초기 정답76문항**으로 유지율을 분석합니다.
초기 형식실패2문항은 전체 형식 점검에만 포함합니다. 새 R2는 **930응답 / 59배치**입니다.
초기 오답109개 후보 중32개는 짧은 연속 구간의 추출 기준을 통과하지 못했습니다.

최대5 자동 문장 단위(표제·목록 조각 포함), 최대180단어 기준이며 실제는 **중앙값4문장·63단어 / 최대5문장·154단어**입니다.
수식은 문장 수와 별도로 기록합니다. 전체 입력은 최대1258토큰으로 기존4096한도 안입니다.
새 요약문을 생성하지 않고, 표기 정리된 원문에서 문단·수식을 중간에 자르지 않은 연속 구간을 뽑습니다.
영어 검수는 필수 실행 단계가 아닙니다. 다만 핵심 논리의 보존과 근거의 충분성은 자동 규칙만으로 보증하지 못합니다.

이전 **full/짧은 판의 R2**와는 풀이가 다릅니다. 이번 max5 R2만 새 저장 폴더에서 생성하며 이전 결과를 덮어쓰지 않습니다.
새 방어·RAG·추가 모델은 없습니다. 탐색적 실험이며 실제 A100 추론은 제작 환경에서 실행하지 않았습니다.


## 1. 설치 — 완료 후 세션 다시 시작
A100을 선택한 뒤 이 셀만 실행하세요. 설치·Qwen3 import 검사 완료 후 **런타임 → 세션 다시 시작**을 누르고 2번부터 실행합니다.


In [ ]:
import os, sys, subprocess
INSTALL_RAN_IN_THIS_KERNEL = True
for key, value in {"TOKENIZERS_PARALLELISM": "false", "HF_HUB_DISABLE_XET": "1",
                   "CUBLAS_WORKSPACE_CONFIG": ":4096:8", "USE_TF": "0", "USE_FLAX": "0"}.items():
    os.environ[key] = value
packages = ["transformers==4.57.6", "accelerate==1.12.0", "bitsandbytes==0.48.2",
            "huggingface-hub==0.36.0", "tokenizers==0.22.2", "numpy==2.2.6", "pandas==2.2.3",
            "pyarrow==20.0.0", "matplotlib==3.10.3", "scipy==1.15.3", "tqdm==4.67.1"]
print("1/3 패키지 설치 중", flush=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("2/3 Transformers 설치 파일 확인·복구 중", flush=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
                       "--no-deps", "--no-cache-dir", "transformers==4.57.6"])
print("3/3 별도 Python 프로세스에서 실제 Qwen3 import 검사", flush=True)
PROBE = """
from transformers.generation import GenerationMixin
from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM
from transformers import BitsAndBytesConfig
import torch, accelerate, bitsandbytes, transformers
print("Qwen3 / GenerationMixin / 양자화 import OK")
print("torch:", torch.__version__, "transformers:", transformers.__version__)
"""
result = subprocess.run([sys.executable, "-c", PROBE], capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("설치/import 검사 실패. 이후 셀로 넘어가지 말고 출력을 공유하세요.")
print("설치 및 import 점검 완료. 지금 세션을 다시 시작하고 2번부터 실행하세요.", flush=True)


## 2. Drive 연결·max5 ZIP 업로드
**sycophancy_qwen_8b_evidenceB_max5_inputs.zip** 하나를 올리세요. 압축 해제, 예전 결과 ZIP, API 키는 필요 없습니다.
최초 답변·기존 R1·원문·발췌·코드가 들어 있습니다. 이후 같은 계정에서는 Drive 캐시를 불러옵니다.


In [ ]:
import os, sys, hashlib, io, zipfile, json, importlib
from pathlib import Path
if globals().get("INSTALL_RAN_IN_THIS_KERNEL", False):
    raise RuntimeError("런타임 → 세션 다시 시작을 누른 뒤 2번부터 실행하세요.")
for key, value in {"TOKENIZERS_PARALLELISM": "false", "HF_HUB_DISABLE_XET": "1",
                    "CUBLAS_WORKSPACE_CONFIG": ":4096:8", "USE_TF": "0", "USE_FLAX": "0"}.items():
    os.environ[key] = value
from google.colab import drive, files
drive.mount("/content/drive")
INPUT_ZIP_NAME = "sycophancy_qwen_8b_evidenceB_max5_inputs.zip"
INPUT_SHA256 = "4264f5277cea80b295b0441997dd0be24cde92850194aa15d36e661d064729c0"
CACHE_DIR = Path("/content/drive/MyDrive/sycophancy_qwen/inputs") / INPUT_SHA256[:12]
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = CACHE_DIR / INPUT_ZIP_NAME
if ZIP_PATH.exists():
    raw = ZIP_PATH.read_bytes()
    if hashlib.sha256(raw).hexdigest() != INPUT_SHA256:
        raise RuntimeError("Drive의 자료 ZIP이 손상됐습니다. 이 오류 출력을 공유하세요.")
    print("Drive에 보관된 자료 ZIP을 불러왔습니다. 다시 업로드할 필요 없습니다.")
else:
    print("아래 파일 선택 버튼에서", INPUT_ZIP_NAME, "하나를 선택하세요.", flush=True)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("자료 ZIP 하나만 선택하고 2번을 다시 실행하세요.")
    raw = bytes(next(iter(uploaded.values())))
    if hashlib.sha256(raw).hexdigest() != INPUT_SHA256:
        raise RuntimeError("지정한 자료 ZIP과 내용이 다릅니다. 이번 inputs.zip을 선택하고 2번을 다시 실행하세요.")
    temp = ZIP_PATH.with_suffix(".zip.tmp")
    temp.write_bytes(raw)
    if hashlib.sha256(temp.read_bytes()).hexdigest() != INPUT_SHA256:
        raise RuntimeError("Drive 저장 검사 실패. 저장 공간과 연결을 확인하고 2번을 다시 실행하세요.")
    temp.replace(ZIP_PATH)
    print("자료 ZIP을 Drive에 보관했습니다. 다음부터 자동으로 불러옵니다.")
CODE_DIR = Path("/content") / ("qwen8b_evidenceB_max5_" + INPUT_SHA256[:12])
CODE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    if any(not (CODE_DIR / name).resolve().is_relative_to(CODE_DIR.resolve()) for name in z.namelist()):
        raise RuntimeError("잘못된 ZIP 내부 경로")
    z.extractall(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
for name in ["protocol", "engine", "analysis", "runtime_check", "progress_ui",
             "ablation_protocol", "normalize_full", "legacy_excerpt", "sentence_units", "extract_evidence", "evidence_protocol", "evidence_run", "evidence_analysis", "length_analysis"]:
    sys.modules.pop(name, None)
importlib.invalidate_caches()
from runtime_check import check_runtime
RUNTIME = check_runtime()
from evidence_protocol import load_source, default_config, freeze, DEFENSES
from evidence_run import EvidenceBackend, bind_backend, run_followups, scan_saved, export_results, write_status
from progress_ui import activity
from protocol import read_json
import pandas as pd
from IPython.display import display, Image
print("자료 준비 완료. 3번부터 실행하세요.")


## 3. 자료 검사·실험 고정·길이 확인
정상 출력: **전체155 / 초기 오답77 / 초기 정답76 / 초기 형식실패2 / 새930 / 기존 재사용930 / 59배치**.
길이 분포를 표시하고 문항별 길이를 저장합니다. 예전 '2문장99개, 3문장118개'는 전체 문항 수였으며 초기 오답 수는 각각53,64였습니다.
이번에는 표기 정리와 목록 단위 검사도 적용했으므로 이전 숫자와의 차이가 문장 제한 하나 때문인 것은 아닙니다.


In [ ]:
from evidence_protocol import load_manifest, selected_controls, DEFENSE_LABELS
from length_analysis import prepare_length_reports
with activity("기존 답변·최대5문장 발췌 자료 검사"):
    SOURCE = load_source(CODE_DIR / "source")
    MANIFEST = load_manifest()
CONFIG = default_config(SOURCE)
STUDY_ROOT = Path("/content/drive/MyDrive/sycophancy_qwen/sycophancy_qwen3_8b_evidenceB_max5_930_a100_b16_v3")
with activity("새 max5 R2 목록·생성 설정·분석 계획 동결"):
    FROZEN, JOBS = freeze(STUDY_ROOT, SOURCE, CONFIG)
display(pd.DataFrame(MANIFEST["availability"]))
print("155문항: 초기 오답77 / 초기 정답76 / 초기 형식실패2")
print("정답 수정률: 같은 77문항 × 3표현 = 조건당 231응답")
print("정답 유지율: 같은 76문항 × 3표현 = 조건당 228응답")
print("기존 R1 유지율은 두 방어 모두100%입니다. 새 오류가 생기는지 보는 보조 분석입니다.")
print("기존 R1 재사용:", len(selected_controls(SOURCE)), "| 신규 R2:", len(JOBS), "| 59배치")
print("최초 답변·R1 재생성 0. 이전 full/짧은 판의 R2는 이번과 합치지 않습니다.")
print("전체 입력 최대:", MANIFEST["token_validation"]["max_input_tokens"], "/4096토큰; 자동 절단 없음")
print("5문장 기준은 자동 집계이며 수식 블록은 별도입니다. 핵심 논리의 보존·충분성을 사람이 검증하지 않았습니다.")
LENGTHS, LENGTH_SUMMARY, LENGTH_COUNTS = prepare_length_reports(STUDY_ROOT, MANIFEST)
display(LENGTH_SUMMARY.rename(columns={"measure":"길이 기준", "min":"최소", "median":"중앙값", "max":"최대"}))
display(LENGTH_COUNTS.rename(columns={"sentence_bin":"문장 수", "wrong":"초기 오답", "correct":"초기 정답", "invalid":"형식 실패", "total":"합계"}))
print("Drive 저장:", STUDY_ROOT)
print("동결 해시:", FROZEN["study_hash"])


## 4. 모델 준비·입력 길이 검사
고정 Qwen3-8B 가중치를 자동 다운로드합니다. A100 / NF4 / BF16 / 비사고 / 배치16 / 생성최대192토큰을 유지합니다.
30초 간격으로 대기 경과를 표시합니다. 입력 자동 절단·배치 자동 축소는 하지 않습니다.


In [ ]:
with activity("저장 배치 확인"):
    SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
if len(SAVED) == 59:
    BACKEND = None
    print("이미 930개가 저장돼 모델 로딩을 건너뜁니다. 7번에서 분석할 수 있습니다.")
else:
    if globals().get("BACKEND") is None:
        with activity("Qwen3-8B 가중치 다운로드·A100 적재"):
            BACKEND = EvidenceBackend(SOURCE, CONFIG)
    bind_backend(STUDY_ROOT, FROZEN, SOURCE, BACKEND, JOBS)
    print("GPU:", BACKEND.environment["gpu"])
    print("원본과 모델/생성 설정 일치. NF4 / BF16 / 비사고 / 배치 16 / 최대 192토큰")
    if BACKEND.source_environment_notes:
        print("기록한 기타 환경 차이:", BACKEND.source_environment_notes)
    print("모델 및 전체 입력 점검 완료. 5번으로 진행하세요.")


## 5. 첫16응답 생성·중간 저장 확인
본실험 첫 배치입니다. 저장·해시 검사 통과 후6번으로 갑니다. 형식 실패도 기록하며 재생성하지 않습니다.


In [ ]:
SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
if 0 not in SAVED:
    STATUS = run_followups(STUDY_ROOT, FROZEN, SOURCE, CONFIG, JOBS, BACKEND, max_new_batches=1)
    SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
else:
    print("첫 배치는 이미 저장되어 재사용합니다.")
from evidence_analysis import score_records
FIRST_CHECK = score_records(SOURCE, SAVED[0]["records"])
print("첫 배치 Drive 저장·해시 검사 통과: 16/16개")
print("형식 실패:", int(FIRST_CHECK.invalid.sum()), "/16 | 잘림:", int(FIRST_CHECK.truncated.sum()), "/16")
print("현재 전체 저장:", sum(len(x["records"]) for x in SAVED.values()), "/930")
if FIRST_CHECK.invalid.all():
    raise RuntimeError("첫 배치 16개가 모두 형식 실패입니다. 저장 자료는 보존하고 이 출력을 공유하세요.")
print("6번을 실행하면 저장된 배치를 건너뛰고 이어갑니다.")


## 6. 나머지 생성
완료 기준은 **930/930 · 59/59 · complete=true**입니다. 16개마다 Drive 저장하며 마지막 배치는2개입니다.
같은 런타임에서 중단되면 이 셀을 다시 실행합니다. 새 런타임에서는1번 설치·재시작 후2번부터 실행합니다. 완료 배치는 건너뜁니다.


In [ ]:
if BACKEND is None:
    SAVED = scan_saved(STUDY_ROOT, JOBS, CONFIG)
    if len(SAVED) != 59:
        raise RuntimeError("4번 모델 준비부터 다시 실행하세요.")
    STATUS = write_status(STUDY_ROOT, SAVED, JOBS, CONFIG, "complete")
    print("이미 모든 배치가 저장되어 재생성하지 않습니다.")
else:
    STATUS = run_followups(STUDY_ROOT, FROZEN, SOURCE, CONFIG, JOBS, BACKEND)
    if STATUS["complete"]:
        BACKEND.close()
        BACKEND = None
print(json.dumps(STATUS, ensure_ascii=False, indent=2))


## 7. 자동 분석
같은 초기 오답77문항에서 수정률, 초기 정답76문항에서 유지율을 각각 비교합니다. 형식 실패를 분모에서 빼지 않습니다.
주 비교는 **(R2 S1−S0)−(R1 S1−S0)** 하나입니다. 표현3개를 문항 안에서 평균해 과목 내 대응 부트스트랩20000회·95% 구간을 구합니다.
1~5문장별 결과도 보조 분석으로 저장합니다. 작은 구간의 유리한 결과만 고르거나 길이의 인과효과로 해석하지 않습니다.
R2 수정률이 낮아도“타당한 근거를 거부했다”고 단정하지 않습니다. 발췌에서 빠진 맥락이나 불충분한 설명도 원인일 수 있습니다.


In [ ]:
from evidence_analysis import analyze
with activity("대응 문항 자동 채점·근거와 방어의 상호작용·그래프"):
    REPORT_DIR, SUMMARY, EFFECTS, ASSESSMENT = analyze(STUDY_ROOT, FROZEN, SOURCE, CONFIG, JOBS)
print("주 분석: 같은 초기 오답 77문항의 네 조건 비교 (%)")
TABLE = pd.read_csv(REPORT_DIR / "comparison_table.csv")
display(TABLE.rename(columns={"condition":"후속 조건", "defense":"방어", "n_items":"문항수",
    "n_responses":"응답수", "correct_count":"정답 수정 수", "final_correct_percent":"정답 수정률",
    "initial_wrong_retained_percent":"초기 오답 유지율", "other_wrong_change_percent":"다른 오답",
    "invalid_percent":"형식 실패"}).round(2))
print("보조 분석: 같은 초기 정답 76문항의 정답 유지 (%)")
RETAINED = pd.read_csv(REPORT_DIR / "retention_table.csv")
display(RETAINED.rename(columns={"condition":"후속 조건", "defense":"방어", "n_items":"문항수",
    "n_responses":"응답수", "correct_retained_count":"정답 유지 수", "correct_retained_percent":"정답 유지율",
    "changed_wrong_percent":"오답 변경률", "invalid_percent":"형식 실패"}).round(2))
print("주 비교: (R2 S1−S0) − (R1 S1−S0). 단위 %포인트, 문항 기준 95% 구간")
display(EFFECTS[EFFECTS.analysis_role == "primary"][["label_ko","n_items","difference_pp","ci_low_pp","ci_high_pp"]].round(2))
print("나머지 비교는 기술적 보조 분석이며 성공 판정에 따로 사용하지 않습니다.")
display(EFFECTS[EFFECTS.analysis_role != "primary"][["label_ko","n_items","difference_pp","ci_low_pp","ci_high_pp"]].round(2))
print(ASSESSMENT["message_ko"])
print("구간에 0이 있으면 효과 없음이나 동등성으로 확정하지 않습니다.")
display(pd.read_csv(REPORT_DIR / "tokens_and_format.csv"))
for name in ["01_evidence_comparison.png","02_revision_outcomes.png","03_correct_retention.png"]:
    display(Image(filename=str(REPORT_DIR / name)))
print("분석 저장:",REPORT_DIR)
from length_analysis import analyze_lengths
with activity("저장 응답의 문장 수별 보조 분석 — 추가 생성 없음"):
    LENGTH_RESULTS, LENGTH_EFFECTS = analyze_lengths(STUDY_ROOT, MANIFEST, CONFIG)
print("문장 수별 수정률·유지율. 같은 구간의 같은 문항을 네 조건에서 비교합니다.")
display(LENGTH_RESULTS.pivot(index=["metric","sentence_bin","n_items","small_bin_under_20"],
                            columns=["condition","defense"],values="correct_percent").round(2))
print("20문항 미만은 small_bin_under_20=True. 길이 구간 간 차이는 길이 자체의 인과효과가 아닙니다.")
print("문항별 길이: explanation_lengths.csv / 구간별 비율: length_group_results.csv / 대응 차이와 구간: length_group_effects.csv")


## 8. 결과 ZIP 저장·다운로드
결과: `sycophancy_qwen3_8b_evidenceB_max5_930_a100_b16_v3_results.zip`.
Drive 경로: `내 드라이브/sycophancy_qwen/sycophancy_qwen3_8b_evidenceB_max5_930_a100_b16_v3/`.
원응답·원문·발췌 위치·설정·표·그래프를 함께 저장합니다. 다운로드가 안 되면 Drive에서 직접 받으세요.


In [ ]:
with activity("결과 ZIP을 Drive에 저장"):
    RESULT_ZIP = export_results(STUDY_ROOT)
print("완료 상태:", read_json(STUDY_ROOT / "status.json") if (STUDY_ROOT / "status.json").exists() else "미실행")
files.download(str(RESULT_ZIP))
